In [4]:
!pip install transformers torch

In [1]:
# importing required libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# loading pretrained DialoGPT model
model_name = "microsoft/DialoGPT-medium"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!\n")

# variable to store conversation history
chat_history_ids = None

# function to generate chatbot response
def generate_response(user_input):
    global chat_history_ids

    # adding simple context so model understands conversation roles
    input_text = "User: " + user_input + " Assistant:"

    # converting text to tokens
    new_input_ids = tokenizer.encode(input_text + tokenizer.eos_token, return_tensors='pt')

    # combining with previous conversation
    if chat_history_ids is not None:
        input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        input_ids = new_input_ids

    attention_mask = torch.ones(input_ids.shape, dtype=torch.long)

    # generating response
    chat_history_ids = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=100,
        do_sample=True,              # adds randomness (more natural replies)
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.2,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )

    # decoding only the new response
    response = tokenizer.decode(
        chat_history_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )
    return response

# starting chatbot interaction
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.\n")

while True:
    user_input = input("You: ")

    # exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a nice day.")
        break
    # generate reply
    reply = generate_response(user_input)
    print("Chatbot:", reply)

Loading model...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model loaded successfully!

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.



You:  what is machine learning?


Chatbot: I think they mean machine programming.


You:  exit


Chatbot: Goodbye! Have a nice day.
